In [1]:
import sqlparse

In [12]:
query = "SELECT account_id, SUM(amount) AS total FROM transactions WHERE account_id = 123 GROUP BY account_id"

In [13]:
sqlparse.parse(query, encoding='utf-8')[0]

<Statement 'SELECT...' at 0x24F31A8A9D0>

---

### sql lineage

In [5]:
from sqllineage.runner import LineageRunner

In [10]:
query = "SELECT account_id, SUM(amount) AS total FROM transactions WHERE account_id = 123 GROUP BY account_id"

In [11]:
print(LineageRunner(query))

Statements(#): 1
Source Tables:
    <default>.transactions
Target Tables:
    



---

### moz_sql_parser

In [14]:
from moz_sql_parser import parse

ImportError: cannot import name 'Iterable' from 'collections' (c:\Users\USER\miniconda3\envs\obaca_env\Lib\collections\__init__.py)

missing expected call export("mo_parsing.core", regex_parameters)
Exception in thread Thread-14 (worker):
Traceback (most recent call last):
  File "c:\Users\USER\miniconda3\envs\obaca_env\Lib\threading.py", line 1045, in _bootstrap_inner
    self.run()
  File "c:\Users\USER\miniconda3\envs\obaca_env\Lib\site-packages\ipykernel\ipkernel.py", line 766, in run_closure
    _threading_Thread_run(self)
  File "c:\Users\USER\miniconda3\envs\obaca_env\Lib\threading.py", line 982, in run
    self._target(*self._args, **self._kwargs)
  File "c:\Users\USER\miniconda3\envs\obaca_env\Lib\site-packages\mo_imports\__init__.py", line 204, in worker
    _error("Missing export() calls")
  File "c:\Users\USER\miniconda3\envs\obaca_env\Lib\site-packages\mo_imports\__init__.py", line 211, in _error
    raise Exception(description)
Exception: Missing export() calls


---

### slqglot

In [26]:
import sqlglot

In [27]:
print(repr(sqlglot.parse_one(query)))

Select(
  expressions=[
    Column(
      this=Identifier(this=account_id, quoted=False)),
    Alias(
      this=Sum(
        this=Column(
          this=Identifier(this=amount, quoted=False))),
      alias=Identifier(this=total, quoted=False))],
  from=From(
    this=Table(
      this=Identifier(this=transactions, quoted=False))),
  where=Where(
    this=EQ(
      this=Column(
        this=Identifier(this=account_id, quoted=False)),
      expression=Literal(this=123, is_string=False))),
  group=Group(
    expressions=[
      Column(
        this=Identifier(this=account_id, quoted=False))]))


In [1]:
import sqlglot
from sqlglot import parse_one, exp

# Original SQL query
sql = "SELECT id, name, age FROM users WHERE age > 30"

# Parse SQL into AST
ast = parse_one(sql)

print("Original AST:")
print(ast)

# Function to remove a column (by name) from SELECT
def remove_column(ast, column_name):
    select_expressions = ast.expressions
    ast.set(
        "expressions",
        [expr for expr in select_expressions if not (isinstance(expr, exp.Column) and expr.name == column_name)]
    )

# Remove 'age' column
remove_column(ast, "age")

# Convert AST back to SQL
modified_sql = ast.sql()

print("\nModified SQL:")
print(modified_sql)

Original AST:
SELECT id, name, age FROM users WHERE age > 30

Modified SQL:
SELECT id, name FROM users WHERE age > 30


In [12]:
import uuid
from sqlglot import parse_one, exp

def decorate_ast_with_ids(ast_node):
    """
    Traverses the AST and assigns a unique ID to a 'meta' attribute on each node.
    
    Args:
        ast_node: The root node of the sqlglot AST.
        
    Returns:
        The same AST with nodes decorated with IDs.
    """
    for node in ast_node.walk():
        # .walk() conveniently visits every node in the tree.
        if not hasattr(node, 'meta'):
            node.meta = {}
        node.meta['id'] = str(uuid.uuid4())[:8]
    return ast_node

In [13]:
def get_node_children(node):
    """Helper to get all child expression nodes from a node's args."""
    for value in node.args.values():
        if isinstance(value, exp.Expression):
            yield value
        elif isinstance(value, list):
            for item in value:
                if isinstance(item, exp.Expression):
                    yield item

def pretty_print_ast(node, prefix="", is_last=True):
    """
    Recursively prints a formatted tree of the AST, including our custom IDs.
    """
    # Use the first 8 characters of the UUID for cleaner printing
    short_id = node.meta.get('id', 'N/A')[:8]
    
    # Get the node's type and any direct value (like a column name)
    node_name = node.__class__.__name__
    if hasattr(node, 'this'):
        node_name += f" ({node.this})"

    # Print the current node with a tree-like structure
    connector = "└── " if is_last else "├── "
    print(f"{prefix}{connector}{node_name}  [ID: {short_id}]")

    # Recurse for children
    children = list(get_node_children(node))
    for i, child in enumerate(children):
        new_prefix = prefix + ("    " if is_last else "│   ")
        pretty_print_ast(child, prefix=new_prefix, is_last=(i == len(children) - 1))

In [16]:
sql = "SELECT c.name, SUM(o.amount) FROM customers AS c JOIN orders AS o ON c.id = o.customer_id WHERE o.amount > 100 AND c.status = 'active' GROUP BY c.name, o.amount"

# 1. Parse the SQL
ast = parse_one(sql)

# 2. Add unique IDs to every node
decorated_ast = decorate_ast_with_ids(ast)

# 3. Print the visual tree
print("--- Decorated AST Visual ---")
pretty_print_ast(decorated_ast)

--- Decorated AST Visual ---
└── Select (None)  [ID: 6e729326]
    ├── Column (name)  [ID: 3089ba32]
    │   ├── Identifier (name)  [ID: 9fba6441]
    │   └── Identifier (c)  [ID: 12bdc566]
    ├── Sum (o.amount)  [ID: 586cb3f0]
    │   └── Column (amount)  [ID: 92fadd12]
    │       ├── Identifier (amount)  [ID: 29ea2c7b]
    │       └── Identifier (o)  [ID: 14fefee6]
    ├── From (customers AS c)  [ID: ca70a586]
    │   └── Table (customers)  [ID: af26f116]
    │       ├── Identifier (customers)  [ID: 28f66c21]
    │       └── TableAlias (c)  [ID: b5ea21ad]
    │           └── Identifier (c)  [ID: 896ee705]
    ├── Join (orders AS o)  [ID: 691de602]
    │   ├── Table (orders)  [ID: db09ef00]
    │   │   ├── Identifier (orders)  [ID: 8c560018]
    │   │   └── TableAlias (o)  [ID: d7d4a983]
    │   │       └── Identifier (o)  [ID: 74585269]
    │   └── EQ (c.id)  [ID: d4c0cfe9]
    │       ├── Column (id)  [ID: 9ffa8eb8]
    │       │   ├── Identifier (id)  [ID: a25a63f1]
    │       │

In [7]:
def remove_node_by_id(ast_node, target_id: str):
    """
    Removes a single node from the AST based on its unique 'meta.id'.

    This function uses ast.transform to walk the tree and delete the
    target node by returning None when it's found.

    Args:
        ast_node: The root node of the decorated sqlglot AST.
        target_id: The unique ID of the node to remove.

    Returns:
        A new AST with the specified node removed.
    """
    if not target_id:
        return ast_node

    # Define the transformer function inside this scope so it has access to 'target_id'.
    # This is a common pattern known as a closure.
    def transformer(node):
        # Check if the node has the ID we're looking for.
        if hasattr(node, 'meta') and node.meta.get('id') == target_id:
            # By returning None, we tell ast.transform to delete this node.
            # sqlglot correctly handles restructuring the parent.
            print(f"--> Found and removing node '{node.__class__.__name__}' with ID: {target_id}")
            return None
        
        # For all other nodes, return them unchanged to keep them in the tree.
        return node

    # Apply the configured transformer to the entire AST.
    return ast_node.transform(transformer)

In [ ]:
round_node_id = ast.find(exp.Sum).meta['id']
ast_after_removal = remove_node_by_id(decorated_ast, "2213aacb")
ast_after_removal.sql()


--> Found and removing node 'GT' with ID: 2213aacb


"SELECT c.name, SUM(o.amount) FROM customers AS c JOIN orders AS o ON c.id = o.customer_id WHERE  AND c.status = 'active'"

In [17]:

round_node_id = ast.find(exp.Sum).meta['id']
ast_after_removal = remove_node_by_id(decorated_ast, "37271798")
ast_after_removal.sql()

--> Found and removing node 'Column' with ID: 37271798


"SELECT c.name, SUM(o.amount) FROM customers AS c JOIN orders AS o ON c.id = o.customer_id WHERE o.amount > 100 AND c.status = 'active' GROUP BY o.amount"

---

### slqgpt_parser

In [15]:
from sqlgpt_parser.parser.mysql_parser import parser as mysql_parser

In [35]:
ast = mysql_parser.parse(query)
ast

Query(query_body=QuerySpecification(select=Select(distinct=False, select_items=[SingleColumn(alias=[], expression=QualifiedNameReference(name=QualifiedName.of("account_id"))), SingleColumn(alias=['AS', 'total'], expression=AggregateFunc(name='SUM', arguments=[QualifiedNameReference(name=QualifiedName.of("amount"))]))]), from_=Table(name=QualifiedName.of("transactions"), for_update=False), where=ComparisonExpression(type='=', left=QualifiedNameReference(name=QualifiedName.of("account_id")), right=LongLiteral(value=123)), group_by=SimpleGroupBy(columns=[ByItem(item=QualifiedNameReference(name=QualifiedName.of("account_id")))]), order_by=[], limit=0, offset=0, for_update=False, nowait_or_wait=False), order_by=[], limit=0, offset=0)

In [19]:
ast.query_body 

QuerySpecification(select=Select(distinct=False, select_items=[SingleColumn(alias=[], expression=QualifiedNameReference(name=QualifiedName.of("account_id"))), SingleColumn(alias=['AS', 'total'], expression=AggregateFunc(name='SUM', arguments=[QualifiedNameReference(name=QualifiedName.of("amount"))]))]), from_=Table(name=QualifiedName.of("transactions"), for_update=False), where=ComparisonExpression(type='=', left=QualifiedNameReference(name=QualifiedName.of("account_id")), right=LongLiteral(value=123)), group_by=SimpleGroupBy(columns=[ByItem(item=QualifiedNameReference(name=QualifiedName.of("account_id")))]), order_by=[], limit=0, offset=0, for_update=False, nowait_or_wait=False)

In [20]:
ast.query_body.where

ComparisonExpression(type='=', left=QualifiedNameReference(name=QualifiedName.of("account_id")), right=LongLiteral(value=123))

In [32]:
ast.query_body.where.left.name

QualifiedName.of("account_id")

In [36]:
ast.query_body.where.left.name.id = "c0039"

In [37]:
ast.query_body.where.left.name

QualifiedName.of("account_id")

In [38]:
ast.query_body.where.left.name.id

'c0039'

In [34]:
ast.query_body.from_

Table(name=QualifiedName.of("transactions"), for_update=False)

In [21]:
from sqlgpt_parser.format.formatter import format_sql

In [23]:
format_sql(ast).replace("\n", " ").strip()

'SELECT   account_id , SUM(amount) AS total FROM   transactions WHERE account_id = 123 GROUP BY account_id'

In [61]:
from sqlglot import parse_one, exp


In [42]:
query1 = "SELECT Orders.OrderID, Customers.CustomerName, Orders.OrderDate FROM Orders INNER JOIN Customers ON Orders.CustomerID=Customers.CustomerID"
ast1 = sqlglot.parse_one(query1)
ast1

Select(
  expressions=[
    Column(
      this=Identifier(this=OrderID, quoted=False),
      table=Identifier(this=Orders, quoted=False)),
    Column(
      this=Identifier(this=CustomerName, quoted=False),
      table=Identifier(this=Customers, quoted=False)),
    Column(
      this=Identifier(this=OrderDate, quoted=False),
      table=Identifier(this=Orders, quoted=False))],
  from=From(
    this=Table(
      this=Identifier(this=Orders, quoted=False))),
  joins=[
    Join(
      this=Table(
        this=Identifier(this=Customers, quoted=False)),
      kind=INNER,
      on=EQ(
        this=Column(
          this=Identifier(this=CustomerID, quoted=False),
          table=Identifier(this=Orders, quoted=False)),
        expression=Column(
          this=Identifier(this=CustomerID, quoted=False),
          table=Identifier(this=Customers, quoted=False))))])

In [64]:
def walk_with_parent(node, parent=None):
    yield parent, node
    for arg_name, child in node.args.items():
        if isinstance(child, exp.Expression):
            yield from walk_with_parent(child, node)
        elif isinstance(child, list):
            for item in child:
                if isinstance(item, exp.Expression):
                    yield from walk_with_parent(item, node)

# Example SQL
sql = "SELECT name, age FROM users WHERE age > 30 AND active = TRUE"
ast = parse_one(sql)

# Walk the tree with parent-child relationships
for parent, child in walk_with_parent(ast):
    print(f"Parent: {type(parent).__name__ if parent else None}, Child: {type(child).__name__}, SQL: {child.sql()}")

Parent: None, Child: Select, SQL: SELECT name, age FROM users WHERE age > 30 AND active = TRUE
Parent: Select, Child: Column, SQL: name
Parent: Column, Child: Identifier, SQL: name
Parent: Select, Child: Column, SQL: age
Parent: Column, Child: Identifier, SQL: age
Parent: Select, Child: From, SQL: FROM users
Parent: From, Child: Table, SQL: users
Parent: Table, Child: Identifier, SQL: users
Parent: Select, Child: Where, SQL: WHERE age > 30 AND active = TRUE
Parent: Where, Child: And, SQL: age > 30 AND active = TRUE
Parent: And, Child: GT, SQL: age > 30
Parent: GT, Child: Column, SQL: age
Parent: Column, Child: Identifier, SQL: age
Parent: GT, Child: Literal, SQL: 30
Parent: And, Child: EQ, SQL: active = TRUE
Parent: EQ, Child: Column, SQL: active
Parent: Column, Child: Identifier, SQL: active
Parent: EQ, Child: Boolean, SQL: TRUE


In [67]:
from sqlglot import parse_one, exp

class TreeNode:
    def __init__(self, sqlglot_node, parent=None):
        self.sqlglot_node = sqlglot_node
        self.parent = parent
        self.children = []

    def add_child(self, child_node):
        self.children.append(child_node)

    def __repr__(self):
        return f"<TreeNode {type(self.sqlglot_node).__name__}: {self.sqlglot_node.sql()}>"

    def walk(self):
        """Yield self and all descendants."""
        yield self
        for child in self.children:
            yield from child.walk()

    def print_tree(self, level=0):
        print("  " * level + repr(self))
        for child in self.children:
            child.print_tree(level + 1)

def build_tree(sqlglot_node, parent_node=None):
    current = TreeNode(sqlglot_node, parent=parent_node)

    for _, child in sqlglot_node.args.items():
        if isinstance(child, exp.Expression):
            child_node = build_tree(child, current)
            current.add_child(child_node)
        elif isinstance(child, list):
            for item in child:
                if isinstance(item, exp.Expression):
                    child_node = build_tree(item, current)
                    current.add_child(child_node)

    return current


In [73]:

# Example usage
sql = "SELECT id, SUM(amount) AS total_amount FROM users WHERE age > 30 AND active = TRUE, GROUP BY id ORDER BY id LIMIT 10"
ast = parse_one(sql)
tree_root = build_tree(ast)

# Print the tree
tree_root.print_tree()


<TreeNode Select: SELECT id, SUM(amount) AS total_amount FROM users WHERE age > 30 AND active = TRUE GROUP BY id ORDER BY id LIMIT 10>
  <TreeNode Column: id>
    <TreeNode Identifier: id>
  <TreeNode Alias: SUM(amount) AS total_amount>
    <TreeNode Sum: SUM(amount)>
      <TreeNode Column: amount>
        <TreeNode Identifier: amount>
    <TreeNode Identifier: total_amount>
  <TreeNode Limit: LIMIT 10>
    <TreeNode Literal: 10>
  <TreeNode From: FROM users>
    <TreeNode Table: users>
      <TreeNode Identifier: users>
  <TreeNode Where: WHERE age > 30 AND active = TRUE>
    <TreeNode And: age > 30 AND active = TRUE>
      <TreeNode GT: age > 30>
        <TreeNode Column: age>
          <TreeNode Identifier: age>
        <TreeNode Literal: 30>
      <TreeNode EQ: active = TRUE>
        <TreeNode Column: active>
          <TreeNode Identifier: active>
        <TreeNode Boolean: TRUE>
  <TreeNode Group: GROUP BY id>
    <TreeNode Column: id>
      <TreeNode Identifier: id>
  <TreeNode

In [71]:
query1 = "SELECT Orders.OrderID, Customers.CustomerName, Orders.OrderDate FROM Orders INNER JOIN Customers ON Orders.CustomerID=Customers.CustomerID"

ast1 = sqlglot.parse_one(query1)
tree_root1 = build_tree(ast1)
# Print the tree for the first query
tree_root1.print_tree()

<TreeNode Select: SELECT Orders.OrderID, Customers.CustomerName, Orders.OrderDate FROM Orders INNER JOIN Customers ON Orders.CustomerID = Customers.CustomerID>
  <TreeNode Column: Orders.OrderID>
    <TreeNode Identifier: OrderID>
    <TreeNode Identifier: Orders>
  <TreeNode Column: Customers.CustomerName>
    <TreeNode Identifier: CustomerName>
    <TreeNode Identifier: Customers>
  <TreeNode Column: Orders.OrderDate>
    <TreeNode Identifier: OrderDate>
    <TreeNode Identifier: Orders>
  <TreeNode From: FROM Orders>
    <TreeNode Table: Orders>
      <TreeNode Identifier: Orders>
  <TreeNode Join: INNER JOIN Customers ON Orders.CustomerID = Customers.CustomerID>
    <TreeNode Table: Customers>
      <TreeNode Identifier: Customers>
    <TreeNode EQ: Orders.CustomerID = Customers.CustomerID>
      <TreeNode Column: Orders.CustomerID>
        <TreeNode Identifier: CustomerID>
        <TreeNode Identifier: Orders>
      <TreeNode Column: Customers.CustomerID>
        <TreeNode Identif

In [72]:
query2 = "SELECT * FROM Orders INNER JOIN Customers ON Orders.CustomerID=Customers.CustomerID"

ast2 = sqlglot.parse_one(query2)
tree_root2 = build_tree(ast2)
# Print the tree for the first query
tree_root2.print_tree()

<TreeNode Select: SELECT * FROM Orders INNER JOIN Customers ON Orders.CustomerID = Customers.CustomerID>
  <TreeNode Star: *>
  <TreeNode From: FROM Orders>
    <TreeNode Table: Orders>
      <TreeNode Identifier: Orders>
  <TreeNode Join: INNER JOIN Customers ON Orders.CustomerID = Customers.CustomerID>
    <TreeNode Table: Customers>
      <TreeNode Identifier: Customers>
    <TreeNode EQ: Orders.CustomerID = Customers.CustomerID>
      <TreeNode Column: Orders.CustomerID>
        <TreeNode Identifier: CustomerID>
        <TreeNode Identifier: Orders>
      <TreeNode Column: Customers.CustomerID>
        <TreeNode Identifier: CustomerID>
        <TreeNode Identifier: Customers>


In [69]:
insert_query = "INSERT INTO users (name, age) VALUES ('Alice', 30)"
insert_ast = parse_one(insert_query)

insert_tree_root = build_tree(insert_ast)
# Print the insert tree
insert_tree_root.print_tree()

<TreeNode Insert: INSERT INTO users (name, age) VALUES ('Alice', 30)>
  <TreeNode Schema: users (name, age)>
    <TreeNode Table: users>
      <TreeNode Identifier: users>
    <TreeNode Identifier: name>
    <TreeNode Identifier: age>
  <TreeNode Values: VALUES ('Alice', 30)>
    <TreeNode Tuple: ('Alice', 30)>
      <TreeNode Literal: 'Alice'>
      <TreeNode Literal: 30>


In [70]:
update_query = "UPDATE users SET age = 31 WHERE name = 'Alice'"

update_ast = parse_one(update_query)
update_tree_root = build_tree(update_ast)
# Print the update tree
update_tree_root.print_tree()


<TreeNode Update: UPDATE users SET age = 31 WHERE name = 'Alice'>
  <TreeNode Table: users>
    <TreeNode Identifier: users>
  <TreeNode EQ: age = 31>
    <TreeNode Column: age>
      <TreeNode Identifier: age>
    <TreeNode Literal: 31>
  <TreeNode Where: WHERE name = 'Alice'>
    <TreeNode EQ: name = 'Alice'>
      <TreeNode Column: name>
        <TreeNode Identifier: name>
      <TreeNode Literal: 'Alice'>


In [41]:
fquery = """
WITH recent_orders AS (
    SELECT
        o.customer_id,
        o.order_id,
        o.order_date,
        RANK() OVER (PARTITION BY o.customer_id ORDER BY o.order_date DESC) AS order_rank
    FROM orders o
    WHERE o.order_date > CURRENT_DATE - INTERVAL '1 year'
),
high_value_customers AS (
    SELECT customer_id
    FROM orders
    GROUP BY customer_id
    HAVING SUM(total_amount) > 10000
)
SELECT
    c.customer_id,
    c.first_name || ' ' || c.last_name AS full_name,
    COALESCE(c.email, 'no_email@example.com') AS email,
    CASE
        WHEN c.vip_status = TRUE THEN 'VIP'
        WHEN hc.customer_id IS NOT NULL THEN 'High Value'
        ELSE 'Regular'
    END AS customer_type,
    COUNT(DISTINCT ro.order_id) AS recent_orders_count,
    AVG(oi.quantity * oi.unit_price) AS avg_item_value,
    ARRAY_AGG(DISTINCT p.category) FILTER (WHERE p.category IS NOT NULL) AS product_categories,
    EXISTS (
        SELECT 1
        FROM support_tickets st
        WHERE st.customer_id = c.customer_id AND st.status = 'open'
    ) AS has_open_tickets,
    (
        SELECT MAX(review_date)
        FROM product_reviews pr
        WHERE pr.customer_id = c.customer_id
    ) AS last_review_date
FROM customers c
LEFT JOIN recent_orders ro ON c.customer_id = ro.customer_id
LEFT JOIN high_value_customers hc ON c.customer_id = hc.customer_id
LEFT JOIN order_items oi ON ro.order_id = oi.order_id
LEFT JOIN products p ON oi.product_id = p.product_id
WHERE c.signup_date BETWEEN DATE '2020-01-01' AND CURRENT_DATE
  AND c.status IN ('active', 'pending')
  AND c.customer_id NOT IN (SELECT customer_id FROM blacklisted_customers)
GROUP BY GROUPING SETS (
    (c.customer_id, c.first_name, c.last_name, c.email, c.vip_status, hc.customer_id),
    ()
)
HAVING COUNT(ro.order_id) > 1 OR SUM(oi.quantity) > 5
ORDER BY customer_type DESC, recent_orders_count DESC
LIMIT 100
OFFSET 10
"""

print(repr(sqlglot.parse_one(fquery)))

Select(
  expressions=[
    Column(
      this=Identifier(this=customer_id, quoted=False),
      table=Identifier(this=c, quoted=False)),
    Alias(
      this=DPipe(
        this=DPipe(
          this=Column(
            this=Identifier(this=first_name, quoted=False),
            table=Identifier(this=c, quoted=False)),
          expression=Literal(this=' ', is_string=True),
          safe=True),
        expression=Column(
          this=Identifier(this=last_name, quoted=False),
          table=Identifier(this=c, quoted=False)),
        safe=True),
      alias=Identifier(this=full_name, quoted=False)),
    Alias(
      this=Coalesce(
        this=Column(
          this=Identifier(this=email, quoted=False),
          table=Identifier(this=c, quoted=False)),
        expressions=[
          Literal(this='no_email@example.com', is_string=True)]),
      alias=Identifier(this=email, quoted=False)),
    Alias(
      this=Case(
        ifs=[
          If(
            this=EQ(
              th

---

### sqlglot 

In [107]:
from sqlglot import parse_one, exp
import uuid 

# --- Type Mapper ---

CLAUSE_TYPES = {
    "From": "FromClause",
    "Group": "GroupByClause",
    "Into": "IntoClause",
    "Limit": "LimitClause",
    "Select": "SelectClause",
    "Set": "SetClause",
    "Update": "UpdateClause",
    "Values": "ValuesClause",
    "Where": "WhereClause"
}

STATEMENT_TYPES = {
    "Delete": "DeleteStatement",
    "Insert": "InsertStatement",
    "Update": "UpdateStatement",
    "Select": "SelectStatement"
}

EXPRESSION_CATEGORIES = {
    "Alias": "Alias",
    "Table": "TableRef",
    "Column": "ColumnRef",
    "Wildcard": "WildCard",
    "Identifier": "Identifier",
    # Functions
    "Sum": "Function",
    "Count": "Function",
    "Avg": "Function",
    "Max": "Function",
    "Min": "Function",
    "Concat": "Function",
    "Coalesce": "Function",
    # Operators
    "And": "Operator",
    "Or": "Operator",
    "EQ": "Operator",
    "GT": "Operator",
    "LT": "Operator",
    "Add": "Operator",
    "Sub": "Operator",
    "Mul": "Operator",
    "Div": "Operator"
}

def map_node_type(node):
    class_name = type(node).__name__
    if class_name in STATEMENT_TYPES:
        return "Statement", STATEMENT_TYPES[class_name]
    elif class_name in CLAUSE_TYPES:
        return "Clause", CLAUSE_TYPES[class_name]
    elif class_name in EXPRESSION_CATEGORIES:
        return EXPRESSION_CATEGORIES[class_name], class_name
    else:
        return "Expression", class_name

# --- TreeNode Class ---

class TreeNode:
    def __init__(self, sqlglot_node, parent=None):
        self.id = "n"+str(uuid.uuid4())[:6]
        self.parent = parent
        self.children = []
        self.sqlglot_node = sqlglot_node
        self.kind, self.name = map_node_type(sqlglot_node)

        if self.name == "Column":
            self.refcol = None
            self.reftable = None
            parts = sqlglot_node.parts
            if len(parts) == 2:
                self.reftable, self.refcol = parts
            elif len(parts) == 1:
                self.refcol = parts[0]

        if self.name == "Table":
            self.reftable = sqlglot_node.name
        
        if self.name == "Alias":
            alias_expr = sqlglot_node.args.get("alias")
            if isinstance(alias_expr, exp.Identifier):
                self.alias = alias_expr.name

    def add_child(self, child):
        self.children.append(child)

    def __repr__(self):
        suffix = f" ({self.name})"
        if self.name == "Column":
            suffix += f" [reftable={self.reftable}, refcol={self.refcol}]"
        if self.name == "Table":
            suffix += f" [reftable={self.reftable}]"
        if self.name == "Alias" and hasattr(self, "alias"):
            suffix += f" [name={self.alias}]"
        return f"<TreeNode ID: {self.id} | {self.kind}{suffix}>"

    def walk(self):
        yield self
        for child in self.children:
            yield from child.walk()

    def print_tree(self, level=0):
        print("  " * level + repr(self))
        for child in self.children:
            child.print_tree(level + 1)

# --- Tree Builder ---

def build_tree(sqlglot_node):
    kind, name = map_node_type(sqlglot_node)
    stmt_node = TreeNode(sqlglot_node)
    stmt_node.kind = "Statement"
    stmt_node.name = name

    clause_root = TreeNode(sqlglot_node, parent=stmt_node)
    clause_root.kind = "Clause"
    clause_root.name = CLAUSE_TYPES.get(type(sqlglot_node).__name__, type(sqlglot_node).__name__ + "Clause")
    stmt_node.add_child(clause_root)

    found_first_clause = False
    for key, child in sqlglot_node.args.items():
        if child is None:
            continue

        children = child if isinstance(child, list) else [child]

        for expr in children:
            if not isinstance(expr, exp.Expression):
                continue

            c_kind, c_name = map_node_type(expr)

            if not found_first_clause and (key != "from" and c_kind != "Clause"):
                sub = _build_recursive(expr, clause_root)
                if sub:
                    clause_root.add_child(sub)
            else:
                found_first_clause = True
                clause_node = TreeNode(expr, parent=stmt_node)
                clause_node.kind = "Clause"
                clause_node.name = c_name
                stmt_node.add_child(clause_node)

                for _, grandchild in expr.args.items():
                    if isinstance(grandchild, exp.Expression):
                        if isinstance(expr, exp.Table) and isinstance(grandchild, exp.Identifier):
                            continue  # Skip Identifier under Table
                        child_node = _build_recursive(grandchild, clause_node)
                        if child_node:
                            clause_node.add_child(child_node)
                    elif isinstance(grandchild, list):
                        for item in grandchild:
                            if isinstance(expr, exp.Table) and isinstance(item, exp.Identifier):
                                continue  # Skip Identifier under Table
                            if isinstance(item, exp.Expression):
                                child_node = _build_recursive(item, clause_node)
                                if child_node:
                                    clause_node.add_child(child_node)

    return stmt_node


def _build_recursive(sqlglot_node, parent_node):
    if isinstance(sqlglot_node, exp.Identifier) and isinstance(parent_node, TreeNode) and parent_node.name == "ColumnRef":
        return None

    if isinstance(parent_node, TreeNode) and parent_node.name == "TableRef" and isinstance(sqlglot_node, exp.Identifier):
        return None

    current = TreeNode(sqlglot_node, parent_node)

    if isinstance(sqlglot_node, exp.Select):
        clause_root = TreeNode(sqlglot_node, parent=current)
        clause_root.kind = "Clause"
        clause_root.name = "SelectClause"
        current.add_child(clause_root)

        found_first_clause = False
        for key, child in sqlglot_node.args.items():
            if child is None:
                continue

            children = child if isinstance(child, list) else [child]
            for expr in children:
                if not isinstance(expr, exp.Expression):
                    continue
                c_kind, c_name = map_node_type(expr)

                if not found_first_clause and (key != "from" and c_kind != "Clause"):
                    sub = _build_recursive(expr, clause_root)
                    if sub:
                        clause_root.add_child(sub)
                else:
                    found_first_clause = True
                    clause_node = TreeNode(expr, parent=current)
                    clause_node.kind = "Clause"
                    clause_node.name = c_name
                    current.add_child(clause_node)

                    for _, grandchild in expr.args.items():
                        if isinstance(grandchild, exp.Expression):
                            if isinstance(expr, exp.Table) and isinstance(grandchild, exp.Identifier):
                                continue
                            child_node = _build_recursive(grandchild, clause_node)
                            if child_node:
                                clause_node.add_child(child_node)
                        elif isinstance(grandchild, list):
                            for item in grandchild:
                                if isinstance(expr, exp.Table) and isinstance(item, exp.Identifier):
                                    continue
                                if isinstance(item, exp.Expression):
                                    child_node = _build_recursive(item, clause_node)
                                    if child_node:
                                        clause_node.add_child(child_node)
    else:
        for _, child in sqlglot_node.args.items():
            if isinstance(child, exp.Expression):
                if isinstance(sqlglot_node, exp.Table) and isinstance(child, exp.Identifier):
                    continue
                child_node = _build_recursive(child, current)
                if child_node:
                    current.add_child(child_node)
            elif isinstance(child, list):
                for item in child:
                    if isinstance(sqlglot_node, exp.Table) and isinstance(item, exp.Identifier):
                        continue
                    if isinstance(item, exp.Expression):
                        child_node = _build_recursive(item, current)
                        if child_node:
                            current.add_child(child_node)

    return current


In [111]:
from sqlglot import parse_one, exp
import uuid 

# --- Type Mapper ---

CLAUSE_TYPES = {
    "From": "FromClause",
    "Group": "GroupByClause",
    "Into": "IntoClause",
    "Limit": "LimitClause",
    "Select": "SelectClause",
    "Set": "SetClause",
    "Update": "UpdateClause",
    "Values": "ValuesClause",
    "Where": "WhereClause"
}

STATEMENT_TYPES = {
    "Delete": "DeleteStatement",
    "Insert": "InsertStatement",
    "Update": "UpdateStatement",
    "Select": "SelectStatement"
}

EXPRESSION_CATEGORIES = {
    "Alias": "Alias",
    "Table": "TableRef",
    "Column": "ColumnRef",
    "Wildcard": "WildCard",
    "Identifier": "Identifier",
    # Functions
    "Sum": "Function",
    "Count": "Function",
    "Avg": "Function",
    "Max": "Function",
    "Min": "Function",
    "Concat": "Function",
    "Coalesce": "Function",
    # Operators
    "And": "Operator",
    "Or": "Operator",
    "EQ": "Operator",
    "GT": "Operator",
    "LT": "Operator",
    "Add": "Operator",
    "Sub": "Operator",
    "Mul": "Operator",
    "Div": "Operator",
    # Literals
    "Literal": "Literal"
}

def map_node_type(node):
    class_name = type(node).__name__
    if class_name in STATEMENT_TYPES:
        return "Statement", STATEMENT_TYPES[class_name]
    elif class_name in CLAUSE_TYPES:
        return "Clause", CLAUSE_TYPES[class_name]
    elif class_name in EXPRESSION_CATEGORIES:
        return "Expression", EXPRESSION_CATEGORIES[class_name]
    else:
        return "Expression", class_name

# --- TreeNode Class ---

class TreeNode:
    def __init__(self, sqlglot_node, parent=None):
        self.id = "n"+str(uuid.uuid4())[:6]
        self.parent = parent
        self.children = []
        self.sqlglot_node = sqlglot_node
        self.kind, self.name = map_node_type(sqlglot_node)

        if self.name == "ColumnRef":
            self.refcol = None
            self.reftable = None
            parts = sqlglot_node.parts
            if len(parts) == 2:
                self.reftable, self.refcol = parts
            elif len(parts) == 1:
                self.refcol = parts[0]

        if self.name == "TableRef":
            self.reftable = sqlglot_node.name

        if self.name == "Alias":
            alias_expr = sqlglot_node.args.get("alias")
            if isinstance(alias_expr, exp.Identifier):
                self.alias = alias_expr.name

        if self.name == "Literal":
            self.value = sqlglot_node.this

    def add_child(self, child):
        self.children.append(child)

    def __repr__(self):
        suffix = f" ({self.name})"
        if self.name == "ColumnRef":
            suffix += f" [reftable={self.reftable}, refcol={self.refcol}]"
        if self.name == "TableRef":
            suffix += f" [reftable={self.reftable}]"
        if self.name == "Alias" and hasattr(self, "alias"):
            suffix += f" [name={self.alias}]"
        if self.name == "Literal" and hasattr(self, "value"):
            suffix += f" [value={self.value}]"
        return f"<TreeNode ID: {self.id} | {self.kind}{suffix}>"

    def walk(self):
        yield self
        for child in self.children:
            yield from child.walk()

    def print_tree(self, level=0):
        print("  " * level + repr(self))
        for child in self.children:
            child.print_tree(level + 1)

# --- Tree Builder ---


def build_tree(sqlglot_node):
    kind, name = map_node_type(sqlglot_node)
    stmt_node = TreeNode(sqlglot_node)
    stmt_node.kind = "Statement"
    stmt_node.name = name

    clause_root = TreeNode(sqlglot_node, parent=stmt_node)
    clause_root.kind = "Clause"
    clause_root.name = CLAUSE_TYPES.get(type(sqlglot_node).__name__, type(sqlglot_node).__name__ + "Clause")
    stmt_node.add_child(clause_root)

    found_first_clause = False
    for key, child in sqlglot_node.args.items():
        if child is None:
            continue

        children = child if isinstance(child, list) else [child]

        for expr in children:
            if not isinstance(expr, exp.Expression):
                continue

            c_kind, c_name = map_node_type(expr)

            if not found_first_clause and (key != "from" and c_kind != "Clause"):
                sub = _build_recursive(expr, clause_root)
                if sub:
                    clause_root.add_child(sub)
            else:
                found_first_clause = True
                clause_node = TreeNode(expr, parent=stmt_node)
                clause_node.kind = "Clause"
                clause_node.name = c_name
                stmt_node.add_child(clause_node)

                for _, grandchild in expr.args.items():
                    if isinstance(grandchild, exp.Expression):
                        if isinstance(expr, exp.Table) and isinstance(grandchild, exp.Identifier):
                            continue  # Skip Identifier under Table
                        child_node = _build_recursive(grandchild, clause_node)
                        if child_node:
                            clause_node.add_child(child_node)
                    elif isinstance(grandchild, list):
                        for item in grandchild:
                            if isinstance(expr, exp.Table) and isinstance(item, exp.Identifier):
                                continue  # Skip Identifier under Table
                            if isinstance(item, exp.Expression):
                                child_node = _build_recursive(item, clause_node)
                                if child_node:
                                    clause_node.add_child(child_node)

    return stmt_node


def _build_recursive(sqlglot_node, parent_node):
    if isinstance(sqlglot_node, exp.Identifier) and isinstance(parent_node, TreeNode) and parent_node.name == "ColumnRef":
        return None

    if isinstance(parent_node, TreeNode) and parent_node.name == "TableRef" and isinstance(sqlglot_node, exp.Identifier):
        return None

    current = TreeNode(sqlglot_node, parent_node)

    if isinstance(sqlglot_node, exp.Select):
        clause_root = TreeNode(sqlglot_node, parent=current)
        clause_root.kind = "Clause"
        clause_root.name = "SelectClause"
        current.add_child(clause_root)

        found_first_clause = False
        for key, child in sqlglot_node.args.items():
            if child is None:
                continue

            children = child if isinstance(child, list) else [child]
            for expr in children:
                if not isinstance(expr, exp.Expression):
                    continue
                c_kind, c_name = map_node_type(expr)

                if not found_first_clause and (key != "from" and c_kind != "Clause"):
                    sub = _build_recursive(expr, clause_root)
                    if sub:
                        clause_root.add_child(sub)
                else:
                    found_first_clause = True
                    clause_node = TreeNode(expr, parent=current)
                    clause_node.kind = "Clause"
                    clause_node.name = c_name
                    current.add_child(clause_node)

                    for _, grandchild in expr.args.items():
                        if isinstance(grandchild, exp.Expression):
                            if isinstance(expr, exp.Table) and isinstance(grandchild, exp.Identifier):
                                continue
                            child_node = _build_recursive(grandchild, clause_node)
                            if child_node:
                                clause_node.add_child(child_node)
                        elif isinstance(grandchild, list):
                            for item in grandchild:
                                if isinstance(expr, exp.Table) and isinstance(item, exp.Identifier):
                                    continue
                                if isinstance(item, exp.Expression):
                                    child_node = _build_recursive(item, clause_node)
                                    if child_node:
                                        clause_node.add_child(child_node)
    else:
        for _, child in sqlglot_node.args.items():
            if isinstance(child, exp.Expression):
                if isinstance(sqlglot_node, exp.Table) and isinstance(child, exp.Identifier):
                    continue
                child_node = _build_recursive(child, current)
                if child_node:
                    current.add_child(child_node)
            elif isinstance(child, list):
                for item in child:
                    if isinstance(sqlglot_node, exp.Table) and isinstance(item, exp.Identifier):
                        continue
                    if isinstance(item, exp.Expression):
                        child_node = _build_recursive(item, current)
                        if child_node:
                            current.add_child(child_node)

    return current


In [72]:
from sqlglot import parse_one, exp

# --- Type Mapper ---

CLAUSE_TYPES = {
    "From": "FromClause",
    "Group": "GroupByClause",
    "Into": "IntoClause",
    "Limit": "LimitClause",
    "Select": "SelectClause",
    "Set": "SetClause",
    "Update": "UpdateClause",
    "Values": "ValuesClause",
    "Where": "WhereClause"
}

EXPRESSION_TYPES = {
    "Alias", "Column", "Function", "Literal", "Table", "Identifier", "Wildcard",
    "And", "Or", "EQ", "GT", "LT", "Add", "Sub", "Mul", "Div"
}

def map_node_type(node):
    class_name = type(node).__name__
    if class_name in CLAUSE_TYPES:
        return "Clause", CLAUSE_TYPES[class_name]
    elif class_name in EXPRESSION_TYPES:
        return "Expression", {
            "Table": "TableRef",
            "Column": "ColumnRef",
            "Wildcard": "WildCard",
            "Identifier": "Identifier",
        }.get(class_name, class_name)
    else:
        return "Expression", class_name

# --- TreeNode Class ---

class TreeNode:
    def __init__(self, sqlglot_node, parent=None):
        self.parent = parent
        self.children = []
        self.sqlglot_node = sqlglot_node
        self.kind, self.name = map_node_type(sqlglot_node)

        if self.name == "ColumnRef":
            self.refcol = None
            self.reftable = None
            parts = sqlglot_node.parts
            if len(parts) == 2:
                self.reftable, self.refcol = parts
            elif len(parts) == 1:
                self.refcol = parts[0]

        if self.name == "TableRef":
            self.reftable = sqlglot_node.name

        if self.name == "Alias":
            alias_expr = sqlglot_node.args.get("alias")
            if isinstance(alias_expr, exp.Identifier):
                self.alias = alias_expr.name

    def add_child(self, child):
        self.children.append(child)

    def __repr__(self):
        suffix = f" ({self.name})"
        if self.name == "ColumnRef":
            suffix += f" [reftable={self.reftable}, refcol={self.refcol}]"
        if self.name == "TableRef":
            suffix += f" [reftable={self.reftable}]"
        if self.name == "Alias" and hasattr(self, "alias"):
            suffix += f" [name={self.alias}]"
        return f"<TreeNode {self.kind}{suffix}>"

    def walk(self):
        yield self
        for child in self.children:
            yield from child.walk()

    def print_tree(self, level=0):
        print("  " * level + repr(self))
        for child in self.children:
            child.print_tree(level + 1)


In [91]:
from sqlglot import parse_one, exp

# --- Type Mapper ---

CLAUSE_TYPES = {
    "From": "FromClause",
    "Group": "GroupByClause",
    "Into": "IntoClause",
    "Limit": "LimitClause",
    "Select": "SelectClause",
    "Set": "SetClause",
    "Update": "UpdateClause",
    "Values": "ValuesClause",
    "Where": "WhereClause"
}

STATEMENT_TYPES = {
    "Delete": "DeleteStatement",
    "Insert": "InsertStatement",
    "Update": "UpdateStatement",
    "Select": "SelectStatement"
}

EXPRESSION_TYPES = {
    "Alias", "Column", "Function", "Literal", "Table", "Identifier", "Wildcard",
    "And", "Or", "EQ", "GT", "LT", "Add", "Sub", "Mul", "Div"
}

def map_node_type(node):
    class_name = type(node).__name__
    if class_name in STATEMENT_TYPES:
        return "Statement", STATEMENT_TYPES[class_name]
    elif class_name in CLAUSE_TYPES:
        return "Clause", CLAUSE_TYPES[class_name]
    elif class_name in EXPRESSION_TYPES:
        return "Expression", {
            "Table": "TableRef",
            "Column": "ColumnRef",
            "Wildcard": "WildCard",
            "Identifier": "Identifier",
        }.get(class_name, class_name)
    else:
        return "Expression", class_name

# --- TreeNode Class ---

class TreeNode:
    def __init__(self, sqlglot_node, parent=None):
        self.parent = parent
        self.children = []
        self.sqlglot_node = sqlglot_node
        self.kind, self.name = map_node_type(sqlglot_node)

        if self.name == "ColumnRef":
            self.refcol = None
            self.reftable = None
            parts = sqlglot_node.parts
            if len(parts) == 2:
                self.reftable, self.refcol = parts
            elif len(parts) == 1:
                self.refcol = parts[0]

        if self.name == "TableRef":
            self.reftable = sqlglot_node.name

        if self.name == "Alias":
            alias_expr = sqlglot_node.args.get("alias")
            if isinstance(alias_expr, exp.Identifier):
                self.alias = alias_expr.name

    def add_child(self, child):
        self.children.append(child)

    def __repr__(self):
        suffix = f" ({self.name})"
        if self.name == "ColumnRef":
            suffix += f" [reftable={self.reftable}, refcol={self.refcol}]"
        if self.name == "TableRef":
            suffix += f" [reftable={self.reftable}]"
        return f"<TreeNode {self.kind}{suffix}>"

    def walk(self):
        yield self
        for child in self.children:
            yield from child.walk()

    def print_tree(self, level=0):
        print("  " * level + repr(self))
        for child in self.children:
            child.print_tree(level + 1)

# --- Tree Builder ---

def build_tree(sqlglot_node):
    kind, name = map_node_type(sqlglot_node)
    stmt_node = TreeNode(sqlglot_node)
    stmt_node.kind = "Statement"
    stmt_node.name = name

    clause_root = TreeNode(sqlglot_node, parent=stmt_node)
    clause_root.kind = "Clause"
    clause_root.name = CLAUSE_TYPES.get(type(sqlglot_node).__name__, type(sqlglot_node).__name__ + "Clause")
    stmt_node.add_child(clause_root)

    found_first_clause = False
    for key, child in sqlglot_node.args.items():
        if child is None:
            continue

        children = child if isinstance(child, list) else [child]

        for expr in children:
            if not isinstance(expr, exp.Expression):
                continue

            c_kind, c_name = map_node_type(expr)

            if not found_first_clause and (key != "from" and c_kind != "Clause"):
                sub = _build_recursive(expr, clause_root)
                if sub:
                    clause_root.add_child(sub)
            else:
                found_first_clause = True
                clause_node = TreeNode(expr, parent=stmt_node)
                clause_node.kind = "Clause"
                clause_node.name = c_name
                stmt_node.add_child(clause_node)

                for _, grandchild in expr.args.items():
                    if isinstance(grandchild, exp.Expression):
                        if isinstance(expr, exp.Table) and isinstance(grandchild, exp.Identifier):
                            continue  # Skip Identifier under Table
                        child_node = _build_recursive(grandchild, clause_node)
                        if child_node:
                            clause_node.add_child(child_node)
                    elif isinstance(grandchild, list):
                        for item in grandchild:
                            if isinstance(expr, exp.Table) and isinstance(item, exp.Identifier):
                                continue  # Skip Identifier under Table
                            if isinstance(item, exp.Expression):
                                child_node = _build_recursive(item, clause_node)
                                if child_node:
                                    clause_node.add_child(child_node)

    return stmt_node


def _build_recursive(sqlglot_node, parent_node):
    if isinstance(sqlglot_node, exp.Identifier) and isinstance(parent_node, TreeNode) and parent_node.name == "ColumnRef":
        return None

    if isinstance(parent_node, TreeNode) and parent_node.name == "TableRef" and isinstance(sqlglot_node, exp.Identifier):
        return None

    current = TreeNode(sqlglot_node, parent_node)

    if isinstance(sqlglot_node, exp.Select):
        clause_root = TreeNode(sqlglot_node, parent=current)
        clause_root.kind = "Clause"
        clause_root.name = "SelectClause"
        current.add_child(clause_root)

        found_first_clause = False
        for key, child in sqlglot_node.args.items():
            if child is None:
                continue

            children = child if isinstance(child, list) else [child]
            for expr in children:
                if not isinstance(expr, exp.Expression):
                    continue
                c_kind, c_name = map_node_type(expr)

                if not found_first_clause and (key != "from" and c_kind != "Clause"):
                    sub = _build_recursive(expr, clause_root)
                    if sub:
                        clause_root.add_child(sub)
                else:
                    found_first_clause = True
                    clause_node = TreeNode(expr, parent=current)
                    clause_node.kind = "Clause"
                    clause_node.name = c_name
                    current.add_child(clause_node)

                    for _, grandchild in expr.args.items():
                        if isinstance(grandchild, exp.Expression):
                            if isinstance(expr, exp.Table) and isinstance(grandchild, exp.Identifier):
                                continue
                            child_node = _build_recursive(grandchild, clause_node)
                            if child_node:
                                clause_node.add_child(child_node)
                        elif isinstance(grandchild, list):
                            for item in grandchild:
                                if isinstance(expr, exp.Table) and isinstance(item, exp.Identifier):
                                    continue
                                if isinstance(item, exp.Expression):
                                    child_node = _build_recursive(item, clause_node)
                                    if child_node:
                                        clause_node.add_child(child_node)
    else:
        for _, child in sqlglot_node.args.items():
            if isinstance(child, exp.Expression):
                if isinstance(sqlglot_node, exp.Table) and isinstance(child, exp.Identifier):
                    continue
                child_node = _build_recursive(child, current)
                if child_node:
                    current.add_child(child_node)
            elif isinstance(child, list):
                for item in child:
                    if isinstance(sqlglot_node, exp.Table) and isinstance(item, exp.Identifier):
                        continue
                    if isinstance(item, exp.Expression):
                        child_node = _build_recursive(item, current)
                        if child_node:
                            current.add_child(child_node)

    return current


In [112]:
# Example usage
sql = "SELECT id, SUM(amount) AS total_amount FROM users WHERE age > 30 AND active = TRUE, GROUP BY id ORDER BY id LIMIT 10"
ast = parse_one(sql)
tree_root = build_tree(ast)

# Print the tree
tree_root.print_tree()

<TreeNode ID: nae7960 | Statement (SelectStatement)>
  <TreeNode ID: n7b50e3 | Clause (SelectClause)>
    <TreeNode ID: n81a840 | Expression (ColumnRef) [reftable=None, refcol=id]>
    <TreeNode ID: nd52167 | Expression (Alias) [name=total_amount]>
      <TreeNode ID: ne91989 | Expression (Function)>
        <TreeNode ID: n553983 | Expression (ColumnRef) [reftable=None, refcol=amount]>
      <TreeNode ID: nc45552 | Expression (Identifier)>
  <TreeNode ID: ne6d1b0 | Clause (LimitClause)>
    <TreeNode ID: nd72e6a | Expression (Literal) [value=10]>
  <TreeNode ID: nc22cef | Clause (FromClause)>
    <TreeNode ID: n9620e6 | Expression (TableRef) [reftable=users]>
  <TreeNode ID: n6bdc4b | Clause (WhereClause)>
    <TreeNode ID: n57fe17 | Expression (Operator)>
      <TreeNode ID: n0373a6 | Expression (Operator)>
        <TreeNode ID: n7a3bd5 | Expression (ColumnRef) [reftable=None, refcol=age]>
        <TreeNode ID: nac9cf5 | Expression (Literal) [value=30]>
      <TreeNode ID: n777cf8 | E

In [110]:
ast

Select(
  expressions=[
    Column(
      this=Identifier(this=id, quoted=False)),
    Alias(
      this=Sum(
        this=Column(
          this=Identifier(this=amount, quoted=False))),
      alias=Identifier(this=total_amount, quoted=False))],
  limit=Limit(
    expression=Literal(this=10, is_string=False)),
  from=From(
    this=Table(
      this=Identifier(this=users, quoted=False))),
  where=Where(
    this=And(
      this=GT(
        this=Column(
          this=Identifier(this=age, quoted=False)),
        expression=Literal(this=30, is_string=False)),
      expression=EQ(
        this=Column(
          this=Identifier(this=active, quoted=False)),
        expression=Boolean(this=True)))),
  group=Group(
    expressions=[
      Column(
        this=Identifier(this=id, quoted=False))]),
  order=Order(
    expressions=[
      Ordered(
        this=Column(
          this=Identifier(this=id, quoted=False)),
        nulls_first=True)]))

In [57]:
# Example usage
sql = "SELECT Orders.OrderID, Customers.CustomerName, Orders.OrderDate FROM Orders INNER JOIN Customers ON Orders.CustomerID=Customers.CustomerID"
ast = parse_one(sql)
tree_root = build_tree(ast)

# Print the tree
tree_root.print_tree()

<TreeNode Statement (SelectStatement)>
  <TreeNode Clause (SelectClause)>
    <TreeNode Expression (ColumnRef) [reftable=Orders, refcol=OrderID]>
    <TreeNode Expression (ColumnRef) [reftable=Customers, refcol=CustomerName]>
    <TreeNode Expression (ColumnRef) [reftable=Orders, refcol=OrderDate]>
  <TreeNode Clause (FromClause)>
    <TreeNode Expression (TableRef) [reftable=Orders]>
  <TreeNode Clause (Join)>
    <TreeNode Expression (TableRef) [reftable=Customers]>
    <TreeNode Expression (EQ)>
      <TreeNode Expression (ColumnRef) [reftable=Orders, refcol=CustomerID]>
      <TreeNode Expression (ColumnRef) [reftable=Customers, refcol=CustomerID]>


In [58]:
sql = """
SELECT
    c.customer_id,
    c.name,
    (
        SELECT COUNT(*)
        FROM orders o
        WHERE o.customer_id = c.customer_id
    ) AS total_orders,
    (
        SELECT AVG(o2.total_amount)
        FROM orders o2
        WHERE o2.customer_id = c.customer_id
          AND o2.order_date > CURRENT_DATE - INTERVAL '30 days'
    ) AS avg_recent_order_amount
FROM customers c
WHERE c.status = 'active';

"""

ast = parse_one(sql)
tree_root = build_tree(ast)

# Print the tree
tree_root.print_tree()

<TreeNode Statement (SelectStatement)>
  <TreeNode Clause (SelectClause)>
    <TreeNode Expression (ColumnRef) [reftable=c, refcol=customer_id]>
    <TreeNode Expression (ColumnRef) [reftable=c, refcol=name]>
    <TreeNode Expression (Alias) [name=total_orders]>
      <TreeNode Expression (Subquery)>
        <TreeNode Statement (SelectStatement)>
          <TreeNode Clause (SelectClause)>
            <TreeNode Expression (Count)>
              <TreeNode Expression (Star)>
          <TreeNode Clause (FromClause)>
            <TreeNode Expression (TableRef) [reftable=orders]>
              <TreeNode Expression (TableAlias)>
                <TreeNode Expression (Identifier)>
          <TreeNode Clause (WhereClause)>
            <TreeNode Expression (EQ)>
              <TreeNode Expression (ColumnRef) [reftable=o, refcol=customer_id]>
              <TreeNode Expression (ColumnRef) [reftable=c, refcol=customer_id]>
      <TreeNode Expression (Identifier)>
    <TreeNode Expression (Alias) [n